# Import data from UCI Machine Learning Repository
### Dataset: Online Retail
https://archive.ics.uci.edu/dataset/352/online+retail

In [10]:
pip install ucimlrepo

In [11]:
from ucimlrepo import fetch_ucirepo 
import duckdb
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
df = online_retail.data.original 

In [12]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


# Data Cleaning
Using SQL to clean data to demonstrate capability

In [31]:
# Filtering dataset
df = duckdb.sql(
    """
    WITH main_unfiltered AS(
        SELECT
            Description                                                                             AS description,
            Quantity                                                                                AS quantity,
            CAST(STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)                                   AS invoice_date,
            UnitPrice                                                                               AS unit_price,
            CAST(CustomerID AS STRING)                                                              AS customer_id,
            Country                                                                                 AS country,
            InvoiceNo                                                                               AS invoice_number,
            MAX(CAST(STRPTIME(InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)) OVER ()                      AS global_max_date
        FROM
            df
    ),

    main AS (
        SELECT
            *
        FROM
            main_unfiltered
        WHERE
            invoice_date < DATE_ADD(global_max_date, INTERVAL '-3 MONTH') -- Filter out the last 90 days, this will be used for target variables
            AND description = UPPER(description) -- Filtering to just Products, Products are capitalised and adjustments such as discounts are lowercase
            AND quantity >= 1
            AND unit_price >= 0.01
    ),

    max_invoice_date AS (
        SELECT
            MAX(invoice_date) AS max_date
        FROM 
            main
    ),

    customer_lifespan AS (
        SELECT
            customer_id,
            DATEDIFF('day', MIN(invoice_date), MAX(invoice_date)) AS customer_lifespan_days,
            COUNT(DISTINCT invoice_number) AS life_time_orders,
            SUM(quantity*unit_price) AS total_spend, 
            MAX(invoice_date) AS last_order_date 
        FROM
            main
        WHERE
            customer_id IS NOT NULL
        GROUP BY
            customer_id
    ),

    discount AS (
        SELECT
            customer_id,
            invoice_date,
            SUM(quantity*unit_price) AS discount,
            COUNT(invoice_number) AS number_of_discounts 
        FROM
            main_unfiltered
        WHERE
            description = 'Discount'
            AND invoice_date < DATE_ADD(global_max_date, INTERVAL '-3 MONTH')
        GROUP BY
            customer_id, 
            invoice_date
    ),

    target_variable AS (
        SELECT
            customer_id
        FROM 
            main_unfiltered
        WHERE
            invoice_date >= DATE_ADD(global_max_date, INTERVAL '-3 MONTH') -- Filter to the last 90 days of data
    ),
    
    final AS (
        SELECT 
            a.*,
            b.customer_lifespan_days,
            b.life_time_orders,
            c.discount * -1 AS discount
        FROM 
            main a
        LEFT JOIN
            customer_lifespan b USING (customer_id)
        LEFT JOIN
            discount c USING (customer_id, invoice_date) -- Can sum create the count of discounts per customer then get the sum for total discount and avg discount

    )

    SELECT * FROM final
    """
).df()

df

,description,quantity,invoice_date,unit_price,customer_id,country,invoice_number,global_max_date,customer_lifespan_days,life_time_orders,discount
0,ENAMEL BREAD BIN CREAM,1,2011-02-28,12.75,14056.0,United Kingdom,545188,2011-12-09,177,17,NaN
1,CERAMIC CHERRY CAKE MONEY BANK,12,2011-02-28,1.45,15656.0,United Kingdom,545190,2011-12-09,106,3,NaN
2,CREAM SWEETHEART MINI CHEST,2,2011-02-28,12.75,15656.0,United Kingdom,545190,2011-12-09,106,3,NaN
3,HEART IVORY TRELLIS LARGE,12,2011-02-28,1.65,15656.0,United Kingdom,545190,2011-12-09,106,3,NaN
4,WOODEN FRAME ANTIQUE WHITE,6,2011-02-28,2.95,15656.0,United Kingdom,545190,2011-12-09,106,3,NaN
...,...,...,...,...,...,...,...,...,...,...,...
323433,PARTY BUNTING,6,2011-03-08,4.95,None,United Kingdom,546008,2011-12-09,<NA>,<NA>,NaN
323434,SET OF 2 TEA TOWELS APPLE AND PEARS,3,2011-03-08,2.95,None,United Kingdom,546008,2011-12-09,<NA>,<NA>,NaN
323435,PEG BAG APPLES DESIGN,3,2011-03-08,2.55,None,United Kingdom,546008,2011-12-09,<NA>,<NA>,NaN
323436,REVOLVER WOODEN RULER,1,2011-03-25,1.95,None,United Kingdom,547723,2011-12-09,<NA>,<NA>,NaN
